# Gold Layer DDL — Company Risk Intelligence Platform
### Star schema: dim_company, dim_date, dim_industry, dim_officer + fact_company_risk
### NOTE on constraints: Delta/Unity Catalog PRIMARY KEY / FOREIGN KEY constraints are
 INFORMATIONAL only (not enforced). If your runtime rejects the CONSTRAINT lines,
 delete them — tables still create and work. Uniqueness is guaranteed by the load logic.



In [0]:
# CELL 1 — Create the gold schema
# =========================================================
CATALOG = "company_risk_intelligence_platform"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")
print("gold schema ready")


In [0]:
# =========================================================
# CELL 2 — Dimension tables (run before the fact)
# =========================================================
spark.sql("""
CREATE TABLE IF NOT EXISTS company_risk_intelligence_platform.gold.dim_company (
    company_id       STRING  NOT NULL,
    company_name     STRING,
    company_number   STRING,
    ticker           STRING,
    ticker_safe      STRING,
    company_status   STRING,
    company_type     STRING,
    date_of_creation DATE,
    CONSTRAINT pk_dim_company PRIMARY KEY (company_id)
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS company_risk_intelligence_platform.gold.dim_date (
    date_key     INT     NOT NULL,
    full_date    DATE,
    day          INT,
    month        INT,
    quarter      INT,
    year         INT,
    day_of_week  STRING,
    CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS company_risk_intelligence_platform.gold.dim_industry (
    industry_id   INT     NOT NULL,
    sector_name   STRING,
    industry_name STRING,
    CONSTRAINT pk_dim_industry PRIMARY KEY (industry_id)
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS company_risk_intelligence_platform.gold.dim_officer (
    officer_summary_id STRING NOT NULL,
    company_id         STRING,
    board_size         INT,
    active_officers    INT,
    resigned_officers  INT,
    director_churn     DOUBLE,
    CONSTRAINT pk_dim_officer PRIMARY KEY (officer_summary_id)
) USING DELTA
""")

print("dimension tables created")



In [0]:
# Gold Layer DDL — Company Risk Intelligence Platform
# Star schema: dim_company, dim_date, dim_industry, dim_officer + fact_company_risk
# Paste each block as its own notebook cell, run in order.
#
# NOTE on constraints: Delta/Unity Catalog PRIMARY KEY / FOREIGN KEY constraints are
# INFORMATIONAL only (not enforced). If your runtime rejects the CONSTRAINT lines,
# delete them — tables still create and work. Uniqueness is guaranteed by the load logic.



# =========================================================
# CELL 3 — Fact table (run after dimensions exist)
# =========================================================
spark.sql("""
CREATE TABLE IF NOT EXISTS company_risk_intelligence_platform.gold.fact_company_risk (
    company_id          STRING  NOT NULL,
    score_date_key      INT,
    industry_id         INT,
    officer_summary_id  STRING,

    debt_to_equity      DOUBLE,
    current_ratio       DOUBLE,
    net_margin          DOUBLE,
    revenue_growth      DOUBLE,
    financial_score     DOUBLE,

    annual_volatility   DOUBLE,
    max_drawdown        DOUBLE,
    beta                DOUBLE,
    market_score        DOUBLE,

    company_age_years   DOUBLE,
    accounts_overdue    INT,
    governance_score    DOUBLE,

    news_volume         INT,
    news_sentiment      DOUBLE,
    news_score          DOUBLE,

    risk_score          DOUBLE,
    risk_band           STRING,
    score_date          DATE,

    CONSTRAINT pk_fact_company_risk PRIMARY KEY (company_id),
    CONSTRAINT fk_fact_company
        FOREIGN KEY (company_id)         REFERENCES company_risk_intelligence_platform.gold.dim_company,
    CONSTRAINT fk_fact_date
        FOREIGN KEY (score_date_key)     REFERENCES company_risk_intelligence_platform.gold.dim_date,
    CONSTRAINT fk_fact_industry
        FOREIGN KEY (industry_id)        REFERENCES company_risk_intelligence_platform.gold.dim_industry,
    CONSTRAINT fk_fact_officer
        FOREIGN KEY (officer_summary_id) REFERENCES company_risk_intelligence_platform.gold.dim_officer
) USING DELTA
""")

print("fact_company_risk created")
